# Statistical testing

We apply Friedman's test to check whether results from a model are significantly better than those of other models. We then perform Nemenyi's post hoc test for pairwise comparisons.

In [ ]:
import pandas as pd
from scipy import stats
import scikit_posthocs as sp
import glob
from sklearn.metrics import recall_score, balanced_accuracy_score, accuracy_score
import numpy as np

## Friedman's Test

In [ ]:
dataset = 'test'

results_path_u2_abmil = sorted(glob.glob(f'/path/abmil/uni2/*/results_{dataset}.csv'))
results_path_u2_transmil = sorted(glob.glob(f'/path/transmil/uni2/*/results_{dataset}.csv'))
results_path_v2_abmil = sorted(glob.glob(f'/path/abmil/virchow2/*/results_{dataset}.csv'))
results_path_v2_transmil = sorted(glob.glob(f'/path/transmil/virchow2/*/results_{dataset}.csv'))

accs_v2_abmil, accs_v2_transmil, accs_u2_abmil, accs_u2_transmil = [], [], [], []
b_accs_v2_abmil, b_accs_v2_transmil, b_accs_u2_abmil, b_accs_u2_transmil = [], [], [], []
recalls_v2_abmil_l, recalls_v2_transmil_l, recalls_u2_abmil_l, recalls_u2_transmil_l = [], [], [], []
recalls_v2_abmil_m, recalls_v2_transmil_m, recalls_u2_abmil_m, recalls_u2_transmil_m = [], [], [], []

for u2_abmil, u2_transmil, v2_abmil, v2_transmil in zip(results_path_u2_abmil, results_path_u2_transmil, results_path_v2_abmil, results_path_v2_transmil):
    df_v2_abmil = pd.read_csv(v2_abmil)
    df_v2_transmil = pd.read_csv(v2_transmil)
    df_u2_abmil = pd.read_csv(u2_abmil)
    df_u2_transmil = pd.read_csv(u2_transmil)
    
    accs_v2_abmil.append(accuracy_score(y_true=df_v2_abmil['label'], y_pred=df_v2_abmil['pred']))
    accs_v2_transmil.append(accuracy_score(y_true=df_v2_transmil['label'], y_pred=df_v2_transmil['pred']))
    accs_u2_abmil.append(accuracy_score(y_true=df_u2_abmil['label'], y_pred=df_u2_abmil['pred']))
    accs_u2_transmil.append(accuracy_score(y_true=df_u2_transmil['label'], y_pred=df_u2_transmil['pred']))
    
    b_accs_v2_abmil.append(balanced_accuracy_score(y_true=df_v2_abmil['label'], y_pred=df_v2_abmil['pred']))
    b_accs_v2_transmil.append(balanced_accuracy_score(y_true=df_v2_transmil['label'], y_pred=df_v2_transmil['pred']))
    b_accs_u2_abmil.append(balanced_accuracy_score(y_true=df_u2_abmil['label'], y_pred=df_u2_abmil['pred']))
    b_accs_u2_transmil.append(balanced_accuracy_score(y_true=df_u2_transmil['label'], y_pred=df_u2_transmil['pred']))
    
    r_meta_v2_abmil = recall_score(y_true=df_v2_abmil['label'], y_pred=df_v2_abmil['pred'], average=None)[-1]
    r_meta_v2_transmil = recall_score(y_true=df_v2_transmil['label'], y_pred=df_v2_transmil['pred'], average=None)[-1]
    r_meta_u2_abmil = recall_score(y_true=df_u2_abmil['label'], y_pred=df_u2_abmil['pred'], average=None)[-1]
    r_meta_u2_transmil = recall_score(y_true=df_u2_transmil['label'], y_pred=df_u2_transmil['pred'], average=None)[-1]
    
    recalls_v2_abmil_m.append(r_meta_v2_abmil)
    recalls_v2_transmil_m.append(r_meta_v2_transmil)
    recalls_u2_abmil_m.append(r_meta_u2_abmil)
    recalls_u2_transmil_m.append(r_meta_u2_transmil)

In [3]:
print(f'Accuracies: {stats.friedmanchisquare(accs_v2_abmil, accs_v2_transmil, accs_u2_abmil, accs_u2_transmil)}')
print(f'Balanced accuracies: {stats.friedmanchisquare(b_accs_v2_abmil, b_accs_v2_transmil, b_accs_u2_abmil, b_accs_u2_transmil)}')
print(f'Recall: {stats.friedmanchisquare(recalls_v2_abmil_m, recalls_v2_transmil_m, recalls_u2_abmil_m, recalls_u2_transmil_m)}')

Accuracies: FriedmanchisquareResult(statistic=np.float64(10.729411764705889), pvalue=np.float64(0.013282753174607781))
Balanced accuracies: FriedmanchisquareResult(statistic=np.float64(10.729411764705889), pvalue=np.float64(0.013282753174607781))
Recall: FriedmanchisquareResult(statistic=np.float64(10.729411764705889), pvalue=np.float64(0.013282753174607781))


## Nemenyi's post hoc test

In [4]:
data_acc = np.array([accs_v2_abmil, accs_v2_transmil, accs_u2_abmil, accs_u2_transmil])
data_b_acc = np.array([b_accs_v2_abmil, b_accs_v2_transmil, b_accs_u2_abmil, b_accs_u2_transmil])
data_recall_m = np.array([recalls_v2_abmil_m, recalls_v2_transmil_m, recalls_u2_abmil_m, recalls_u2_transmil_m])

In [6]:
sp.posthoc_nemenyi_friedman(data_acc.T)

,0,1,2,3
0,1.000000,1.000000,0.072451,0.306950
1,1.000000,1.000000,0.072451,0.306950
2,0.072451,0.072451,1.000000,0.899884
3,0.306950,0.306950,0.899884,1.000000
